# GRLP random networks: generate and analyze at scale

The hand-built [5-segment network](example_network.ipynb) showed the mechanics of confluences. This capstone does two things that only make sense at scale:

1. **Generate** a whole river network with no DEM required — a random **Shreve (1966, 1974)** binary tree, wired up with discharges and widths automatically.
2. **Analyze its structure** — Strahler stream orders, Horton ratios, and Hack-type scaling — the descriptors that characterize real drainage networks.

This is the quickest way to get a runnable GRLP network for experiments. Reference: [McNab et al. (2025, ESurf)](https://doi.org/10.5194/esurf-13-1059-2025).

In [ ]:
import random

import numpy as np
from matplotlib import pyplot as plt

import grlp

# Seed both RNGs for a reproducible network; drop these to draw a new one.
random.seed(7)
np.random.seed(7)

## 1. Generate a Shreve random network

`generate_random_network` builds the topology and populates it with node spacing, discharges (accumulating downstream), and valley widths. The main knobs:

* `magnitude` — the number of channel heads (the network's "size").
* `max_length` — the length of the longest source-to-outlet path [m].
* `mean_discharge` — sets segment discharges from drainage area.

It returns the `Network` and its topology object.

In [ ]:
net, topo = grlp.generate_random_network(
    magnitude=8,
    max_length=2.0e4,
    mean_discharge=10.,
)
net.set_niter(3)
net.get_z_lengths()
print('%d segments, %d channel heads'
      % (len(net.list_of_LongProfile_objects),
         len(net.list_of_channel_head_segment_IDs)))

## 2. Evolve to steady state and plot the long profiles

Exactly as for the hand-built network — one call evolves the whole graph.

In [ ]:
net.evolve_threshold_width_river_network(nt=200, dt=3.15e11)

plt.figure(figsize=(10, 6))
for lp in net.list_of_LongProfile_objects:
    if lp.downstream_segment_IDs:
        ds = net.list_of_LongProfile_objects[lp.downstream_segment_IDs[0]]
        plt.plot([lp.x[-1] / 1000., ds.x[0] / 1000.],
                 [lp.z[-1], ds.z[0]], 'k-', lw=1, alpha=0.5)
    else:
        plt.plot([lp.x[-1] / 1000., lp.x_ghost_downstream / 1000.],
                 [lp.z[-1], lp.z_bl], 'k-', lw=1, alpha=0.5)
    plt.plot(lp.x / 1000., lp.z, '-', lw=2)
plt.xlabel('Downstream distance [km]', fontsize=14)
plt.ylabel('Elevation [m]', fontsize=14)
plt.title('Shreve random network at steady state')
plt.tight_layout()
plt.show()

## 3. The network planform

`Network.plot()` draws the branching map view — the dendritic pattern of the random tree.

In [ ]:
net.plot()

## 4. Analyze the network structure

Random Shreve networks reproduce the statistical laws of real drainage networks. GRLP computes the classic descriptors.

**Strahler stream order** — headwater streams are order 1; two streams of order *n* join to make order *n+1*.

In [ ]:
net.compute_strahler_orders()
print('Strahler order of each segment:',
      [int(o) for o in net.segment_orders])

**Horton ratios** — the near-constant ratios between successive stream orders (Horton 1945; Schumm 1956): how quickly stream *number*, *length*, and *discharge* change from one order to the next. (A *stream* of a given order is a maximal chain of same-order segments, so there are fewer streams than segments at the higher orders.)

In [ ]:
net.compute_horton_ratios()
print('Number of streams per order:',
      {int(k): int(v) for k, v in net.order_counts.items()})
print('Bifurcation ratio (R_B): %.2f' % net.bifurcation_ratio)
print('Length ratio      (R_L): %.2f' % net.length_ratio)
print('Discharge ratio   (R_Q): %.2f' % net.discharge_ratio)

**Hack-type scaling** — Hack's law relates downstream distance to upstream drainage (here discharge) as a power law. `find_hack_parameters` fits the coefficient `k` and exponent `p`.

In [ ]:
hack = net.find_hack_parameters()
print('Hack coefficient k = %.4g' % hack['k'])
print('Hack exponent    p = %.3f' % hack['p'])

## Wrap-up

That's the three-step path through GRLP: a single [long profile](example_1d.ipynb), a hand-built [network](example_network.ipynb), and a generated-and-analyzed random network. For larger network studies (including the McNab et al., 2025 experiments) and the full API, see [grlp.readthedocs.io](https://grlp.readthedocs.io).